# Computing Ground Truth Values for Eval Questions

This notebook computes the "correct" answer for each eval question that has a 
verifiable single-value answer. These values get embedded in `eval/questions.json` 
as `expected_result` fields, allowing `evaluate.py` to verify the agent's 
output matches reality — not just that the SQL ran.

For each question, we:
1. State the question
2. State our interpretation of "correct"
3. Run the canonical SQL
4. Record the result

If our interpretation is wrong, the eval will be checking against the wrong 
ground truth. So we're explicit about assumptions.

In [11]:
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / 'data' / 'hmda.db'

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to: {DB_PATH}")

Connected to: c:\Users\kzhang\Documents\Professional\Projects\text-to-sql\data\hmda.db


In [2]:
result = con.execute("""
    SELECT SUM(loan_amount) AS total_originated_volume
    FROM hmda_ny
    WHERE action_taken = 1
""").fetchone()

print(f"Q2 expected_result: {result[0]}")

Q2 expected_result: 116181685000


In [3]:
result = con.execute("""
    SELECT COUNT(*) AS high_income_apps
    FROM hmda_ny
    WHERE income > 200
""").fetchone()

print(f"Q9 expected_result: {result[0]}")

Q9 expected_result: 96103


In [4]:
# Primary check
primary = con.execute("""
    SELECT COUNT(*) FROM hmda_ny
    WHERE co_applicant_sex = 5
""").fetchone()[0]

# Sanity check with a different column
secondary = con.execute("""
    SELECT COUNT(*) FROM hmda_ny
    WHERE co_applicant_ethnicity_1 = 5
""").fetchone()[0]

print(f"Q11 expected_result (via co_applicant_sex):       {primary}")
print(f"Q11 sanity check (via co_applicant_ethnicity_1):  {secondary}")
print(f"Match: {primary == secondary}")

Q11 expected_result (via co_applicant_sex):       245373
Q11 sanity check (via co_applicant_ethnicity_1):  245443
Match: False


In [5]:
result = con.execute("""
    SELECT COUNT(*) AS multi_race_apps
    FROM hmda_ny
    WHERE applicant_race_2 IS NOT NULL
""").fetchone()

print(f"Q12 expected_result: {result[0]}")

# Distribution check — how many races do applicants typically report?
print("\nDistribution of races reported:")
con.execute("""
    SELECT
        CASE
            WHEN applicant_race_2 IS NULL THEN '1 race'
            WHEN applicant_race_3 IS NULL THEN '2 races'
            WHEN applicant_race_4 IS NULL THEN '3 races'
            WHEN applicant_race_5 IS NULL THEN '4 races'
            ELSE '5 races'
        END AS num_races,
        COUNT(*) AS count
    FROM hmda_ny
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

Q12 expected_result: 27252

Distribution of races reported:


,num_races,count
0,1 race,403301
1,2 races,25535
2,3 races,1583
3,4 races,96
4,5 races,38


In [12]:
con.execute("""
        SELECT column_name
        , data_type FROM information_schema.columns 
        WHERE table_name = 'hmda_ny' AND data_type = 'VARCHAR'
""").fetchdf()

,column_name,data_type
0,lei,VARCHAR
1,state_code,VARCHAR
2,co_applicant_ethnicity_5,VARCHAR
3,applicant_age,VARCHAR
4,co_applicant_age,VARCHAR
5,total_points_and_fees,VARCHAR
6,discount_points,VARCHAR
7,lender_credits,VARCHAR
8,prepayment_penalty_term,VARCHAR
9,debt_to_income_ratio,VARCHAR


In [13]:
con.close()
print("Connection closed.")

Connection closed.
